# ARC Prize 2026 (H2) ??Duck-REPL Transfer: an agentic exact-oracle REPL loop for ARC-AGI-2 on Qwen3.8-27B

**Hypothesis branch H2 ??"Duck-REPL Transfer"** (`h2/duck-repl`).

The claim: the mechanism that won ARC-AGI-3 Milestone 1 ??The Duck harness (Tufa Labs),
a tool-using REPL loop where *"the model never presses a key. It writes Python, and the
Python presses keys"* ??transfers to ARC-AGI-2, because ARC-AGI-2 ships **ground-truth
train outputs**, which means an exact, free, deterministic oracle is available at
inference time. The Duck's `action(actions)` host round-trip becomes `verify(fn)`: the
agent writes a candidate transform, executes it against every train pair, gets exact
pass/fail plus diff diagnostics, and iterates.

Supporting evidence: arXiv 2607.15439 ablated exactly this on ARC-AGI-3 and found the
**verification variant ranked first in every setting**. Its cost was 2.8x the baseline
(90.75M vs 32.32M cost-tokens at xhigh); this design lands at ~35.5M for 240 tasks
because the oracle is exact (so it *terminates* search rather than extending it), the
context is ~9.6K tokens/turn rather than ~25-30K, and `reasoning_effort` is pinned off
the `xhigh` default.

Full reasoning, measured context costs, the budget arithmetic and the refutation
criteria are in `HYPOTHESIS.md` in the source repository.

### What this notebook does

1. writes the embedded harness package to `/kaggle/working/h2`
2. installs vLLM from an attached **offline wheelhouse** (no internet at rerun)
3. locates the Qwen3.8-27B weights among the attached inputs, rejecting NVFP4 mirrors
   (NVFP4 is Blackwell-native; L4 is Ada sm_89)
4. launches `vllm serve` with `TP=4`, `--reasoning-parser qwen3`,
   `--enable-auto-tool-choice --tool-call-parser qwen3_coder`,
   `--enable-prefix-caching --mamba-cache-mode align`
5. runs the agentic REPL loop over every test challenge with 16 concurrent workers
6. writes `/kaggle/working/submission.json` **cumulatively and atomically**, pre-filled
   with valid fallbacks for all task ids before the first task starts, and finalised by
   a watchdog before the 600s buffer

### Important

`kernel-metadata.json` ships with **`"enable_gpu": false`** so this notebook can be
pushed and smoke-tested without consuming GPU quota. In that state there is no vLLM
server, so the run falls back to a deterministic mock agent: it still produces a
**schema-valid submission** and exercises every non-LLM code path, but its score is
meaningless. **Flip `enable_gpu` to `true` and attach a vLLM wheelhouse dataset for the
real rerun.**


## 1. Environment, deadline and embedded harness source

Detects the competition rerun, counts GPUs without importing torch, computes the hard deadline (12h minus the 600s buffer), and unzips the harness package that is base64-embedded at the bottom of this cell.

In [ ]:
# --- H2 Duck-REPL: environment, deadline, and harness source -----------------
# Everything this notebook needs is either attached as a Kaggle dataset or
# embedded below, because the competition rerun has no internet.
import base64
import io
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

NOTEBOOK_START = time.time()
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}

# 12h Kaggle limit, minus the 600s buffer the competition requires the
# submission to be written before, minus a reserve for vLLM load + CUDA graph
# capture. The watchdog finalises the submission at HARD_DEADLINE.
TOTAL_SECONDS = 12 * 3600.0
SAFETY_BUFFER = 600.0
STARTUP_RESERVE = 1200.0
HARD_DEADLINE = NOTEBOOK_START + TOTAL_SECONDS - SAFETY_BUFFER
USABLE_SECONDS = TOTAL_SECONDS - SAFETY_BUFFER - STARTUP_RESERVE

WORKING = Path("/kaggle/working") if Path("/kaggle").is_dir() else Path.cwd()
SRC = WORKING / "h2"
SUBMISSION_PATH = WORKING / "submission.json"
LOG_DIR = WORKING / "logs"
for directory in (SRC, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

COMPETITION_SLUG = "arc-prize-2026-arc-agi-2"
DATA_DIR_CANDIDATES = [
    Path(f"/kaggle/input/competitions/{COMPETITION_SLUG}"),
    Path(f"/kaggle/input/{COMPETITION_SLUG}"),
    Path("/kaggle/input/arc-prize-2026-arc-agi-2"),
    Path.cwd() / "data",
]
CHALLENGE_GLOB = "arc-agi_*_challenges.json"


def list_inputs(limit=60):
    """Diagnostic: what actually got mounted. Kaggle's competition mount path has
    moved between releases and a code competition may only expose data at rerun,
    so guessing a single path is fragile -- print what is really there."""
    root = Path("/kaggle/input")
    if not root.is_dir():
        print("  /kaggle/input does not exist (not running on Kaggle?)")
        return []
    entries = []
    for path in sorted(root.iterdir()):
        kind = "dir " if path.is_dir() else "file"
        entries.append(f"  {kind} {path}")
    for line in entries[:limit]:
        print(line)
    if len(entries) > limit:
        print(f"  ... {len(entries) - limit} more")
    return entries


def _has_challenges(directory: Path) -> bool:
    return directory.is_dir() and any(directory.glob(CHALLENGE_GLOB))


def find_data_dir():
    """Locate the ARC challenge files.

    Order: explicit ARC_DATA_DIR (validated -- an override that points somewhere
    empty must fall through rather than be trusted), then the known mount paths,
    then a search of every mounted input. The last step is what makes this
    survive a mount-path change.
    """
    override = os.environ.get("ARC_DATA_DIR", "").strip()
    if override:
        candidate = Path(override)
        if _has_challenges(candidate):
            return candidate
        print(f"WARNING: ARC_DATA_DIR={override} holds no {CHALLENGE_GLOB}; ignoring it")
    for candidate in DATA_DIR_CANDIDATES:
        if _has_challenges(candidate):
            return candidate
    root = Path("/kaggle/input")
    if root.is_dir():
        for hit in root.rglob(CHALLENGE_GLOB):
            return hit.parent
    return None


def find_sample_submission():
    """Last-resort source of the task-id -> n_test mapping."""
    for candidate in (
        Path("/kaggle/input") / f"competitions/{COMPETITION_SLUG}/sample_submission.json",
        Path.cwd() / "sample_submission.json",
    ):
        if candidate.is_file():
            return candidate
    root = Path("/kaggle/input")
    if root.is_dir():
        for hit in root.rglob("sample_submission.json"):
            return hit
    for candidate in DATA_DIR_CANDIDATES:
        path = candidate / "sample_submission.json"
        if path.is_file():
            return path
    return None


def count_gpus():
    """GPU count without importing torch (torch may not be present on a CPU run)."""
    try:
        out = subprocess.run(
            ["nvidia-smi", "-L"], capture_output=True, text=True, timeout=30
        )
        if out.returncode == 0:
            return len([ln for ln in out.stdout.splitlines() if ln.strip().startswith("GPU ")])
    except Exception:
        pass
    return 0


print("--- mounted inputs (/kaggle/input) ---")
list_inputs()
print("--------------------------------------")

DATA_DIR = find_data_dir()
SAMPLE_SUBMISSION = find_sample_submission()
N_GPU = count_gpus()

print(f"competition rerun : {TRUE_SUBMISSION}")
print(f"working dir       : {WORKING}")
print(f"data dir          : {DATA_DIR}")
print(f"sample submission : {SAMPLE_SUBMISSION}")
print(f"gpus visible      : {N_GPU}")
print(f"usable seconds    : {USABLE_SECONDS:,.0f}  hard deadline in {HARD_DEADLINE - time.time():,.0f}s")

SERVED_MODEL_NAME = "qwen3.8-27b"
VLLM_PORT = 8000
VLLM_BASE_URL = f"http://127.0.0.1:{VLLM_PORT}/v1"
# Our measured context ceiling is a 6.4K-token prefix plus a ~5.2K rolling
# window, so 32768 is >3x headroom. Capping here rather than at the upstream
# 262144 default lets vLLM schedule more concurrent sequences, which is what
# makes the decode throughput in HYPOTHESIS.md section 5.3 reachable.
MAX_MODEL_LEN = 32768
MAX_NUM_SEQS = 32
TENSOR_PARALLEL = 4
WORKERS = 16

# --- write the embedded harness source ---------------------------------------
# A zip of harness/*.py, base64'd, so this notebook is self-contained and does
# not depend on a source dataset being attached correctly.
HARNESS_ZIP_B64 = "..."


def materialize_harness():
    if not HARNESS_ZIP_B64.strip():
        raise RuntimeError("harness payload is empty; rebuild the notebook with scripts/build_notebook.py")
    blob = base64.b64decode(HARNESS_ZIP_B64)
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        archive.extractall(SRC)
    files = sorted(p.name for p in (SRC / "harness").glob("*.py"))
    print(f"harness written to {SRC}/harness  ({len(files)} modules, {len(blob):,} bytes)")
    print("  " + ", ".join(files))
    return files


MATERIALIZED = materialize_harness()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import harness  # noqa: E402
from harness import dataio as H_dataio  # noqa: E402

print(f"harness version   : {harness.__version__}")


## 2. Offline runtime install

No internet at rerun, so vLLM has to come from an attached wheelhouse dataset (`pip install --no-index --find-links ...`, the same pattern the Duck's Kaggle notebook uses for the ARC runtime). If neither vLLM nor a GPU is present the notebook degrades to the mock agent instead of failing.

In [ ]:
# --- offline runtime install --------------------------------------------------
# The rerun has no internet. vLLM must come from an attached wheelhouse dataset;
# if it is already importable (or absent and unobtainable) we degrade to a
# CPU-only run that still writes a schema-valid submission.
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

WHEELHOUSE_MARKERS = ("vllm", "wheelhouse", "wheels")


def have(module):
    return importlib.util.find_spec(module) is not None


def find_wheelhouse():
    root = Path("/kaggle/input")
    if not root.is_dir():
        return None
    for marker in WHEELHOUSE_MARKERS:
        for candidate in sorted(root.rglob(marker)):
            if candidate.is_dir() and any(candidate.glob("vllm*.whl")):
                return candidate
    for candidate in sorted(root.rglob("vllm*.whl")):
        return candidate.parent
    return None


def pip_install_offline(wheelhouse, packages):
    cmd = [
        sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
        "--no-warn-conflicts", "--disable-pip-version-check",
        "--find-links", str(wheelhouse), *packages,
    ]
    print("$ " + " ".join(cmd))
    return subprocess.run(cmd, stdout=subprocess.DEVNULL).returncode


RUNTIME = {"vllm": have("vllm"), "torch": have("torch"), "PIL": have("PIL"), "numpy": have("numpy")}
print("importable before install:", RUNTIME)

WHEELHOUSE = find_wheelhouse()
print(f"wheelhouse: {WHEELHOUSE}")

if not RUNTIME["vllm"] and WHEELHOUSE is not None:
    rc = pip_install_offline(WHEELHOUSE, ["vllm"])
    print(f"vllm install rc={rc}")
    RUNTIME["vllm"] = have("vllm")

if not RUNTIME["PIL"]:
    # Pillow renders the panel image. Without it the harness simply runs
    # text-only: build_task_block never depends on render succeeding.
    if WHEELHOUSE is not None:
        pip_install_offline(WHEELHOUSE, ["pillow"])
    RUNTIME["PIL"] = have("PIL")

print("importable after install:", RUNTIME)

# The sandbox needs numpy for model-written code; the harness itself does not.
if not RUNTIME["numpy"]:
    print("WARNING: numpy missing -- model-generated transforms will be restricted to pure Python")

VLLM_AVAILABLE = bool(RUNTIME["vllm"] and N_GPU >= 1)
print(f"vLLM serving possible: {VLLM_AVAILABLE}  (vllm={RUNTIME['vllm']}, gpus={N_GPU})")
if not VLLM_AVAILABLE:
    print(
        "NOTE: falling back to the deterministic mock agent. This produces a "
        "schema-valid submission so the notebook never fails silently, but its "
        "score is meaningless. Attach a vLLM wheelhouse dataset and enable GPU "
        "for a real run."
    )


## 3. Locate the model weights

Scans every mounted input for a HF-style Qwen3 snapshot rather than hardcoding a path, so the weights dataset can be swapped without editing the notebook. Prefers FP8 (~25.3 GB) and **rejects NVFP4 outright**: NVFP4 is Blackwell-native and L4 is Ada (sm_89).

In [ ]:
# --- locate the model weights -------------------------------------------------
# Scans every attached Kaggle input for a HF-style snapshot instead of hardcoding
# a mount path, so the weights dataset can be swapped without editing this
# notebook. NVFP4 mirrors are rejected on sight: NVFP4 is Blackwell-native and
# L4 is Ada (sm_89).
import json
import os
from pathlib import Path

MODEL_HINTS = ("qwen3", "qwen3-8-27b", "qwen3.8", "27b")
NVFP4_MARKERS = ("nvfp4", "modelopt", "fp4")


def looks_like_hf_snapshot(directory: Path) -> bool:
    config = directory / "config.json"
    if not config.is_file():
        return False
    return any((directory / name).exists() for name in ("tokenizer_config.json", "generation_config.json")) or bool(
        list(directory.glob("*.safetensors")) or list(directory.glob("*.json.index"))
    )


def quant_of(directory: Path) -> str:
    name = str(directory).lower()
    for marker in ("nvfp4", "fp4", "awq", "int4", "gptq", "fp8", "bf16", "bnb"):
        if marker in name:
            return marker
    try:
        config = json.loads((directory / "config.json").read_text(encoding="utf-8"))
    except Exception:
        return "unknown"
    text = json.dumps(config.get("quantization_config") or {}).lower()
    for marker in NVFP4_MARKERS + ("awq", "int4", "gptq", "fp8", "compressed-tensors", "bitsandbytes"):
        if marker in text:
            return "nvfp4" if marker in NVFP4_MARKERS else marker
    arch = str(config.get("architectures") or "")
    return "bf16" if "ForCausalLM" in arch or "ForConditionalGeneration" in arch else "unknown"


def size_gb(directory: Path) -> float:
    total = 0
    for path in directory.rglob("*"):
        try:
            if path.is_file() and not path.is_symlink():
                total += path.stat().st_size
        except OSError:
            pass
    return total / (1024 ** 3)


def find_candidates():
    root = Path("/kaggle/input")
    found = []
    if not root.is_dir():
        return found
    for directory in root.rglob("config.json"):
        parent = directory.parent
        if not looks_like_hf_snapshot(parent):
            continue
        arch = ""
        try:
            arch = str(json.loads(directory.read_text(encoding="utf-8")).get("architectures") or "")
        except Exception:
            continue
        if "Qwen3" not in arch and not any(h in str(parent).lower() for h in MODEL_HINTS):
            continue
        quant = quant_of(parent)
        found.append(
            {
                "path": parent,
                "quant": quant,
                "arch": arch,
                "size_gb": round(size_gb(parent), 1),
                "blocked": quant in ("nvfp4", "fp4"),
            }
        )
    seen, unique = set(), []
    for item in found:
        resolved = str(item["path"].resolve())
        if resolved in seen:
            continue
        seen.add(resolved)
        unique.append(item)
    return unique


CANDIDATES = find_candidates()
for item in CANDIDATES:
    flag = "BLOCKED(nvfp4 on Ada)" if item["blocked"] else ""
    print(f"  {item['size_gb']:>6.1f} GB  {item['quant']:<10} {item['arch'][:44]:<44} {item['path']} {flag}")

# Prefer: fp8 (25.3 GB, fits TP=4 comfortably) > bf16 (44 GB) > int4/awq.
# Anything NVFP4 is excluded because L4 is sm_89 and NVFP4 is Blackwell-native.
PREFERENCE = {"fp8": 0, "compressed-tensors": 1, "bf16": 2, "unknown": 3, "awq": 4, "int4": 5, "gptq": 6}
USABLE = [c for c in CANDIDATES if not c["blocked"]]
USABLE.sort(key=lambda c: (PREFERENCE.get(c["quant"], 9), abs(c["size_gb"] - 25.3)))

MODEL_PATH = str(USABLE[0]["path"]) if USABLE else None
MODEL_QUANT = USABLE[0]["quant"] if USABLE else None
print(f"\nselected model : {MODEL_PATH}")
print(f"quantisation   : {MODEL_QUANT}")
if MODEL_PATH is None:
    print("WARNING: no usable Qwen3 weights found among the attached inputs.")
else:
    expected_gb = 25.3 if MODEL_QUANT == "fp8" else None
    if expected_gb and abs(USABLE[0]["size_gb"] - expected_gb) > 8:
        print(f"NOTE: expected ~{expected_gb} GB for fp8, found {USABLE[0]['size_gb']} GB")


## 4. Launch vLLM

The official Qwen3.8 serving pattern. Flags are probed against `--help` first because vLLM renames them between releases and the rerun cannot check online. `--mamba-cache-mode align` is required for prefix caching on this GatedDeltaNet hybrid. Speculative decoding / MTP is left **off** because it breaks constrained decoding. `--max-model-len 32768` rather than 262144: the measured context ceiling is a 6.4K prefix plus a ~5.2K rolling window, and a lower cap lets vLLM schedule more concurrent sequences, which is what makes the decode throughput reachable.

In [ ]:
# --- launch vLLM (offline) ----------------------------------------------------
# Official Qwen3.8 serving pattern, with the traps from HYPOTHESIS.md handled:
#   * --reasoning-parser qwen3 and --tool-call-parser qwen3_coder (native tools)
#   * --mamba-cache-mode align, required for prefix caching on the GatedDeltaNet
#     hybrid; without it prefix caching is unsupported
#   * no --speculative-config: MTP/speculative decoding breaks constrained
#     decoding on this model
#   * reasoning_effort is never set to `high` (HTTP 500, QwenLM/Qwen3.8#217);
#     the client validates against {low, medium, xhigh} and drops the field
#     entirely if the server still 500s
# Flags are probed against --help first, because vLLM renames them between
# releases and the rerun has no internet to check against.
import json
import os
import shlex
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

VLLM_LOG = LOG_DIR / "vllm.log"
SERVER_PROC = None
SERVER_READY = False


def module_args():
    return [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]


def probe_help():
    try:
        out = subprocess.run(
            module_args() + ["--help"], capture_output=True, text=True, timeout=300
        )
        return (out.stdout or "") + (out.stderr or "")
    except Exception as exc:
        print(f"--help probe failed: {exc}")
        return ""


HELP_TEXT = probe_help() if VLLM_AVAILABLE else ""
print(f"--help probe returned {len(HELP_TEXT):,} chars")


def supported(flag):
    return flag in HELP_TEXT


def build_command(tensor_parallel):
    cmd = module_args() + [
        "--model", MODEL_PATH,
        "--served-model-name", SERVED_MODEL_NAME,
        "--tensor-parallel-size", str(tensor_parallel),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--max-num-seqs", str(MAX_NUM_SEQS),
        "--gpu-memory-utilization", "0.90",
        "--trust-remote-code",
        "--port", str(VLLM_PORT),
        "--host", "127.0.0.1",
        "--disable-log-requests",
    ]
    optional = [
        ["--reasoning-parser", "qwen3"],
        ["--enable-auto-tool-choice"],
        ["--tool-call-parser", "qwen3_coder"],
        ["--enable-prefix-caching"],
        ["--mamba-cache-mode", "align"],
    ]
    for flag in optional:
        if not HELP_TEXT or supported(flag[0]):
            cmd.extend(flag)
        else:
            print(f"  skipping unsupported flag {flag[0]}")
    if not HELP_TEXT or supported("--limit-mm-per-prompt"):
        cmd.extend(["--limit-mm-per-prompt", json.dumps({"image": 2})])
    return cmd


def wait_ready(timeout_s, proc):
    url = f"{VLLM_BASE_URL}/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if proc.poll() is not None:
            return False, f"server exited with code {proc.returncode}"
        try:
            with urllib.request.urlopen(url, timeout=10) as response:
                if response.status == 200:
                    payload = json.loads(response.read().decode())
                    ids = [m.get("id") for m in payload.get("data", [])]
                    return True, f"ready, models={ids}"
        except (urllib.error.URLError, ConnectionError, TimeoutError, OSError):
            pass
        time.sleep(5)
    return False, f"not ready after {timeout_s:.0f}s"


def launch():
    global SERVER_PROC, SERVER_READY
    if not VLLM_AVAILABLE or not MODEL_PATH:
        print("skipping vLLM launch (unavailable)")
        return False

    tp = min(TENSOR_PARALLEL, max(1, N_GPU))
    # 27 GB of fp8 weights does not fit in one 24 GB L4, so TP>=2 is mandatory
    # for this model. Prefer the documented TP=4 over a single 96 GB pool.
    if tp < 2:
        print(f"only {tp} GPU(s): FP8 27B needs >=2 (27 GB > 24 GB per L4)")
        return False

    load_budget = min(2400.0, max(600.0, (HARD_DEADLINE - time.time()) * 0.25))
    attempts = [tp]
    if tp == 4:
        attempts.append(2)  # TP=2 x 2 instances halves all-reduce participants
    for attempt_tp in attempts:
        cmd = build_command(attempt_tp)
        print("$ " + " ".join(shlex.quote(c) for c in cmd))
        env = dict(os.environ)
        env["VLLM_LOGGING_LEVEL"] = "INFO"
        env["HF_HUB_OFFLINE"] = "1"
        env["TRANSFORMERS_OFFLINE"] = "1"
        env["TOKENIZERS_PARALLELISM"] = "false"
        handle = open(VLLM_LOG, "ab")
        proc = subprocess.Popen(cmd, stdout=handle, stderr=subprocess.STDOUT, env=env)
        ok, detail = wait_ready(load_budget, proc)
        print(f"TP={attempt_tp}: {detail}")
        if ok:
            SERVER_PROC, SERVER_READY = proc, True
            print(f"vLLM ready in {time.time() - NOTEBOOK_START:.0f}s since notebook start")
            return True
        proc.kill()
        try:
            tail = Path(VLLM_LOG).read_text(encoding="utf-8", errors="replace")[-3000:]
            print("--- vllm.log tail ---\n" + tail)
        except Exception:
            pass
    return False


SERVER_READY = launch()
print(f"\nSERVER_READY = {SERVER_READY}")
print(f"elapsed since notebook start: {time.time() - NOTEBOOK_START:.0f}s")


## 5. Run the agentic REPL loop

14 turns per task target (24 is the break-even at planned serving rates), 16 concurrent workers, ~9.6K prompt tokens per turn. Per-task time and turn allowances are recomputed from `remaining x workers / remaining_tasks` before every dispatch, so a slow start shortens later tasks instead of running out of clock. `xhigh` is never used ??it would cost 28h of decode alone.

In [ ]:
# --- run the Duck-REPL agent over the test challenges -------------------------
# Writes /kaggle/working/submission.json cumulatively: the file is pre-filled
# with valid fallbacks for EVERY task id before the first task starts and is
# rewritten atomically after each task, so a kill at any instant still yields a
# schema-valid submission. A watchdog thread finalises it before the buffer.
import json
import os
import time
from pathlib import Path

from harness.budget import Deadline
from harness.config import HarnessConfig
from harness.dataio import (
    is_competition_rerun,
    iter_tasks,
    load_challenges,
    load_solutions,
    pick_challenge_file,
    task_shapes,
)
from harness.llm import VLLMClient
from harness.mock_agent import build_mock_llm
from harness.runner import Runner, write_run_report
from harness.submission import validate_submission

NO_DATA = DATA_DIR is None
if NO_DATA:
    # No challenge file is reachable. In a real competition rerun Kaggle injects
    # /kaggle/input/competitions/arc-prize-2026-arc-agi-2/ itself, so this path
    # only triggers on an interactive "Save & Run All" whose data sources did not
    # attach. We degrade loudly instead of raising: there is nothing to solve and
    # nothing to submit, and an exception would hide the diagnosis under a
    # papermill traceback. The task ids are deliberately NOT embedded in this
    # notebook -- competition data is not redistributed in the repo.
    print("!! DATA_DIR is None -- no ARC challenge file is reachable")
    if SAMPLE_SUBMISSION is not None and SAMPLE_SUBMISSION.is_file():
        # sample_submission.json carries every task id with the right number of
        # attempt slots, which is enough to leave a schema-valid (zero-scoring)
        # file rather than none at all.
        with open(SAMPLE_SUBMISSION, encoding="utf-8") as handle:
            SAMPLE = json.load(handle)
        FALLBACK = {
            task_id: [
                {"attempt_1": [[0, 0], [0, 0]], "attempt_2": [[0, 0], [0, 0]]}
                for _ in entries
            ]
            for task_id, entries in SAMPLE.items()
        }
        with open(SUBMISSION_PATH, "w", encoding="utf-8") as handle:
            json.dump(FALLBACK, handle)
        print(f"wrote a schema-valid placeholder from {SAMPLE_SUBMISSION.name} "
              f"({len(FALLBACK)} task ids). Score will be ~0; fix the data mount.")
    else:
        print("no sample_submission.json either -- nothing to submit from this run.")
        print("TO FIX: in the Kaggle notebook UI use 'Add input' and attach")
        print(f"  * the competition  : {COMPETITION_SLUG}")
        print("  * the model weights : saltb0x/qwen3-8-27b-fp8")
        print("  * a vLLM wheelhouse dataset (offline install; no internet at rerun)")
        print("then re-run. At competition rerun the data is mounted automatically,")
        print("so this message does not indicate a problem with the rerun path.")
    RESULT = None
    CHALLENGE_FILE = None
    CHALLENGES = {}
    SOLUTIONS = None
    SHAPES = {}
    TASKS = []
else:
    CHALLENGE_FILE = os.environ.get("ARC26_CHALLENGE_FILE", "").strip() or pick_challenge_file(
        str(DATA_DIR), prefer_test=is_competition_rerun()
    )
    CHALLENGES = load_challenges(str(DATA_DIR), CHALLENGE_FILE)
    SOLUTIONS = load_solutions(str(DATA_DIR), CHALLENGE_FILE)
    SHAPES = task_shapes(CHALLENGES)
print(f"challenge file : {CHALLENGE_FILE}")
print(f"tasks          : {len(CHALLENGES)}   test outputs: {sum(SHAPES.values())}")
print(f"solutions      : {'yes (scoring enabled)' if SOLUTIONS else 'no (rerun mode)'}")

# ARC26_LIMIT / ARC26_TASKS exist so a real Kaggle run can be smoke-tested on a
# handful of tasks before burning GPU hours; they are unset in a competition rerun.
LIMIT = int(os.environ.get("ARC26_LIMIT", "0") or 0) or None
ONLY = [t.strip() for t in os.environ.get("ARC26_TASKS", "").split(",") if t.strip()] or None
if os.environ.get("ARC26_MAX_TURNS"):
    config_max_turns = int(os.environ["ARC26_MAX_TURNS"])
else:
    config_max_turns = 14  # design target, HYPOTHESIS.md 5.4; break-even is 24

TASKS = iter_tasks(CHALLENGES, limit=LIMIT, task_ids=ONLY)
print(f"loaded tasks   : {len(TASKS)}" + (f" (limited to {LIMIT})" if LIMIT else ""))

# --- config -------------------------------------------------------------------
config = HarnessConfig(
    model=SERVED_MODEL_NAME,
    base_url=VLLM_BASE_URL,
    api_key="EMPTY",
    workers=WORKERS,
)
config.budget.total_seconds = USABLE_SECONDS
config.budget.safety_buffer_seconds = 0.0      # already folded into the deadline
config.budget.startup_reserve_seconds = 0.0
config.budget.max_turns = config_max_turns
config.budget.hard_max_turns = config_max_turns + 6  # break-even is 24 at plan rates
config.budget.max_task_seconds = 900.0
config.budget.context_window = MAX_MODEL_LEN
config.budget.context_budget = 24_000
config.budget.keep_recent_turns = 6
config.budget.use_image = True
config.budget.image_max_vision_tokens = 160    # Rule 1 of HYPOTHESIS.md section 4
config.sampling.max_tokens = 3_072
# effort_first/default/on_shape_fail default to medium/low/medium: `xhigh` (the
# server default) would emit 8-16K thinking tokens per turn and cost 28h of
# decode alone. Never pass `high` -- it is not a valid value and returns HTTP 500.

elapsed = time.time() - NOTEBOOK_START
remaining = max(60.0, HARD_DEADLINE - time.time() - 120.0)
deadline = Deadline(remaining)
print(f"elapsed {elapsed:.0f}s, agent deadline in {remaining:.0f}s")

if SERVER_READY:
    llm = VLLMClient(
        base_url=config.base_url,
        model=config.model,
        api_key=config.api_key,
        timeout=config.request_timeout_s,
        max_retries=config.max_retries,
    )
    print(f"backend: vLLM at {config.base_url} model={config.model} workers={config.workers}")
else:
    llm = build_mock_llm()
    config.workers = 4
    print("backend: MockLLM (no server) -- schema validation only, score is meaningless")


def log_line(text):
    stamp = time.time() - NOTEBOOK_START
    print(f"[{stamp:7.0f}s] {text}", flush=True)
    try:
        with open(LOG_DIR / "run.log", "a", encoding="utf-8") as handle:
            handle.write(f"{stamp:.0f} {text}\n")
    except Exception:
        pass


RESULT = None
if DATA_DIR is not None and TASKS:
    runner = Runner(
        llm,
        config,
        TASKS,
        submission_path=str(SUBMISSION_PATH),
        task_shapes=SHAPES,
        solutions=SOLUTIONS,
        total_tasks=len(TASKS),
        deadline=deadline,
        workers=config.workers,
        keep_transcript=False,
        log=log_line,
    )
    try:
        RESULT = runner.run()
    finally:
        runner.finalize()
        if SERVER_PROC is not None and SERVER_PROC.poll() is None:
            try:
                SERVER_PROC.terminate()
            except Exception:
                pass
elif DATA_DIR is not None:
    print("!! no tasks to solve -- leaving the pre-filled submission in place")

if RESULT is not None:
    write_run_report(str(WORKING / "run_report.json"), RESULT)
    print(f"\n=== run complete in {RESULT.seconds:.0f}s ===")
    print(f"tasks attempted : {RESULT.tasks_attempted}/{len(TASKS)}")
    print(f"tasks verified  : {RESULT.tasks_verified}")
    total = RESULT.ledger.get("total", {})
    calls = total.get("calls") or 0
    print(f"llm calls       : {calls}   python: {total.get('python_calls')}   verify: {total.get('verify_calls')}")
    if calls:
        print(f"prompt tok/turn : {total.get('prompt_tokens', 0) / calls:,.0f}")
        print(f"compl  tok/turn : {total.get('completion_tokens', 0) / calls:,.0f}")
    print(f"total tokens    : {total.get('total_tokens', 0):,}")
    if RESULT.schema_problems:
        print(f"SCHEMA PROBLEMS : {len(RESULT.schema_problems)} -> {RESULT.schema_problems[:5]}")
    else:
        print("schema          : valid")
    if RESULT.score:
        print(f"SCORE           : {RESULT.score['score'] * 100:.2f}%  "
              f"({RESULT.score['n_correct']}/{RESULT.score['n_outputs']} test outputs)")


## 6. Validate and forecast

Re-validates the file on disk against the challenge file itself (every task id, both attempts, ints 0-9, rectangular), repairs in place if anything is wrong, and turns the measured per-turn cost into a 240-task wall-clock forecast with headroom.

In [ ]:
# --- validate the submission and forecast the 240-task cost --------------------
import json
import os
from pathlib import Path

from harness.dataio import load_challenges, task_shapes
from harness.submission import validate_submission

if not Path(SUBMISSION_PATH).is_file():
    # Only reachable when no data mounted at all: there is nothing to solve and
    # no task ids to build a placeholder from. Report it plainly rather than
    # raising, so the kernel log stays readable.
    print(f"!! {SUBMISSION_PATH} does not exist -- no data was mounted, nothing to validate")
    print(f"total notebook elapsed: {time.time() - NOTEBOOK_START:.0f}s")
    SUBMISSION = {}
    FRESH_SHAPES = {}
    PROBLEMS = []
else:
    with open(SUBMISSION_PATH, encoding="utf-8") as handle:
        SUBMISSION = json.load(handle)
    if CHALLENGE_FILE is None:
        # Placeholder path: validate against sample_submission.json, which still
        # proves the file on disk is structurally what the scorer expects.
        REFERENCE = json.load(open(SAMPLE_SUBMISSION, encoding="utf-8"))
        FRESH_SHAPES = {task_id: len(entries) for task_id, entries in REFERENCE.items()}
        print("validating against sample_submission.json (no challenge file was reachable)")
    else:
        # Re-validate against the challenge file itself, not the in-memory copy,
        # so a stale or partially written file cannot slip through.
        FRESH_SHAPES = task_shapes(load_challenges(str(DATA_DIR), CHALLENGE_FILE))
    PROBLEMS = validate_submission(SUBMISSION, FRESH_SHAPES)

if Path(SUBMISSION_PATH).is_file():
    print(f"submission      : {SUBMISSION_PATH}")
    print(f"size            : {os.path.getsize(SUBMISSION_PATH) / 1e6:.2f} MB")
    print(f"task ids        : {len(SUBMISSION)} / {len(FRESH_SHAPES)} required")
    print(f"test outputs    : {sum(len(v) for v in SUBMISSION.values())} / {sum(FRESH_SHAPES.values())} required")
    print(f"schema problems : {len(PROBLEMS)}")
    for problem in PROBLEMS[:10]:
        print(f"  - {problem}")

    if PROBLEMS:
        # Repair rather than fail: a Kaggle rerun scores what is on disk, and a
        # repaired file beats a valid-looking exception.
        from harness.submission import build_submission, write_submission

        print("REPAIRING submission in place")
        repaired = build_submission(SUBMISSION, FRESH_SHAPES)
        for task_id, entries in repaired.items():
            for entry in entries:
                for key in ("attempt_1", "attempt_2"):
                    if not entry.get(key):
                        entry[key] = [[0, 0], [0, 0]]
        write_submission(SUBMISSION_PATH, repaired)
        PROBLEMS = validate_submission(json.load(open(SUBMISSION_PATH, encoding="utf-8")), FRESH_SHAPES)
        print(f"schema problems after repair: {len(PROBLEMS)}")

    assert not PROBLEMS, f"submission is invalid: {PROBLEMS[:5]}"

    if FRESH_SHAPES:
        sample_id = next(iter(FRESH_SHAPES))
        print(f"\nexample entry ({sample_id}):")
        print(json.dumps(SUBMISSION[sample_id])[:300])

# --- 240-task forecast from measured per-turn cost ----------------------------
if RESULT is not None:
    total = RESULT.ledger.get("total", {})
    calls = total.get("calls") or 0
    if calls and RESULT.tasks_attempted:
        prompt_per_turn = total.get("prompt_tokens", 0) / calls
        completion_per_turn = total.get("completion_tokens", 0) / calls
        turns_per_task = calls / RESULT.tasks_attempted
        PREFILL, DECODE, N_RERUN = 2000.0, 400.0, 240
        per_turn = prompt_per_turn / PREFILL + completion_per_turn / DECODE
        forecast = N_RERUN * turns_per_task * per_turn
        print("\n=== 240-task forecast from measured cost ===")
        print(f"prompt/turn {prompt_per_turn:,.0f}  completion/turn {completion_per_turn:,.0f}  turns/task {turns_per_task:.2f}")
        print(f"GPU-seconds/turn {per_turn:.2f}  (prefill {prompt_per_turn / PREFILL:.2f}s + decode {completion_per_turn / DECODE:.2f}s)")
        print(f"forecast {forecast:,.0f}s = {forecast / 3600:.2f}h vs usable {USABLE_SECONDS / 3600:.2f}h -> headroom {USABLE_SECONDS / max(forecast, 1):.2f}x")
        print(f"max affordable turns/task at these rates: {USABLE_SECONDS / (N_RERUN * max(per_turn, 1e-9)):.1f}")
        if forecast > USABLE_SECONDS:
            print("!! DOES NOT FIT: lower max_turns or reasoning effort before submitting")

print(f"\ntotal notebook elapsed: {time.time() - NOTEBOOK_START:.0f}s of {TOTAL_SECONDS:.0f}s")
print("done")
